# Feature Engineering

In this notebook, we create meaningful features from raw transaction data to improve fraud detection.

### Features:
- Hour of transaction
- Customer age
- Geographic distance (customer → merchant)

In [1]:
import pandas as pd
import numpy as np

In [2]:
train_df = pd.read_csv("../data/raw/fraudTrain.csv")
test_df = pd.read_csv("../data/raw/fraudTest.csv")

In [3]:
for df in [train_df, test_df]:
    df["trans_date_trans_time"] = pd.to_datetime(df["trans_date_trans_time"])
    df["hour"] = df["trans_date_trans_time"].dt.hour

In [4]:
for df in [train_df, test_df]:
    df["dob"] = pd.to_datetime(df["dob"])
    df["age"] = (df["trans_date_trans_time"] - df["dob"]).dt.days // 365

In [5]:
from math import radians, sin, cos, sqrt, atan2

def haversine(lat1, lon1, lat2, lon2):
    R = 6371  # Earth radius in km
    lat1, lon1, lat2, lon2 = map(radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = sin(dlat/2)**2 + cos(lat1) * cos(lat2) * sin(dlon/2)**2
    c = 2 * atan2(sqrt(a), sqrt(1-a))
    
    return R * c

In [6]:
for df in [train_df, test_df]:
    df["geo_distance"] = df.apply(
        lambda x: haversine(x["lat"], x["long"], x["merch_lat"], x["merch_long"]),
        axis=1
    )

In [7]:
drop_cols = [
    "Unnamed: 0", "first", "last", "street", "trans_num",
    "trans_date_trans_time", "merchant", "city", "state",
    "job", "dob", "cc_num"
]
train_df = train_df.drop(columns=drop_cols)
test_df = test_df.drop(columns=drop_cols)
train_df = pd.get_dummies(train_df, columns=["category"])
test_df = pd.get_dummies(test_df, columns=["category"])

In [8]:
train_df["gender"] = train_df["gender"].map({"M": 0, "F": 1})
test_df["gender"] = test_df["gender"].map({"M": 0, "F": 1})

In [9]:
bool_cols = train_df.select_dtypes(include="bool").columns
train_df[bool_cols] = train_df[bool_cols].astype(int)
test_df[bool_cols] = test_df[bool_cols].astype(int)

In [10]:
train_df, test_df = train_df.align(test_df, join="left", axis=1, fill_value=0)

In [11]:
train_df.head()

,amt,gender,zip,lat,long,city_pop,unix_time,merch_lat,merch_long,is_fraud,...,category_grocery_pos,category_health_fitness,category_home,category_kids_pets,category_misc_net,category_misc_pos,category_personal_care,category_shopping_net,category_shopping_pos,category_travel
0,4.97,1,28654,36.0788,-81.1781,3495,1325376018,36.011293,-82.048315,0,...,0,0,0,0,1,0,0,0,0,0
1,107.23,1,99160,48.8878,-118.2105,149,1325376044,49.159047,-118.186462,0,...,1,0,0,0,0,0,0,0,0,0
2,220.11,0,83252,42.1808,-112.2620,4154,1325376051,43.150704,-112.154481,0,...,0,0,0,0,0,0,0,0,0,0
3,45.00,0,59632,46.2306,-112.1138,1939,1325376076,47.034331,-112.561071,0,...,0,0,0,0,0,0,0,0,0,0
4,41.96,0,24433,38.4207,-79.4629,99,1325376186,38.674999,-78.632459,0,...,0,0,0,0,0,1,0,0,0,0


In [12]:
train_df.to_csv("../data/processed/train_processed.csv", index=False)
test_df.to_csv("../data/processed/test_processed.csv", index=False)